<h2 style="color:#D35400; margin-bottom:6px;">
  02 – PyTorch BiLSTM Model (From-Scratch Neural Network)
</h2>

<p style="font-size:16px; margin-top:4px;">
  <strong>Overview:</strong>
  In this notebook, I implement a from-scratch deep learning model for
  multi-label emotion classification using <strong>PyTorch</strong>.
  The architecture is a bidirectional LSTM (BiLSTM) built directly on
  tokenized word indices.
</p>

<p style="font-size:16px;">
  <strong>Goal:</strong>
  To move beyond TF-IDF-based models and learn sequence representations
  end-to-end, allowing the model to capture word order and contextual
  patterns in the input text.
</p>

<p style="font-size:16px;">
  The model uses a custom vocabulary, embedding layer, bidirectional LSTM,
  and a fully connected output layer with sigmoid activation to support
  multi-label predictions over the five target emotions:
  <em>anger, fear, joy, sadness,</em> and <em>surprise</em>.
  Training is stabilized using early stopping based on validation
  Macro F1 score.
</p>


In [3]:
#wandb login check to make sure everything goes smooth

from kaggle_secrets import UserSecretsClient
import wandb

# Load API key from Kaggle Secrets
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("WANDB_API_KEY")

# Login to wandb
wandb.login(key=api_key)

print("W&B login successful!")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sharmilam-official (sharmila-m) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B login successful!


In [4]:
# 04 – PyTorch BiLSTM Model (From-Scratch Neural Network)

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
from collections import Counter
import wandb

# -------------------------
# 0. Config & W&B Init
# -------------------------
label_cols = ["anger", "fear", "joy", "sadness", "surprise"]

wandb.init(
    project="24ds2000048-t32025",
    entity="sharmila-m",
    name="04_pytorch_bilstm_from_scratch",
    config={
        "max_len": 50,
        "embed_dim": 128,
        "hidden_dim": 64,
        "batch_size": 64,
        "lr": 1e-3,
        "n_epochs": 20,
        "patience": 3,
        "val_split": 0.1,
        "threshold": 0.5,
        "random_state": 42
    }
)
config = wandb.config

# For reproducibility
torch.manual_seed(config.random_state)
np.random.seed(config.random_state)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# -------------------------
# 1. Load Data
# -------------------------
train_df = pd.read_csv("/kaggle/input/2025-sep-dl-gen-ai-project/train.csv")
test_df = pd.read_csv("/kaggle/input/2025-sep-dl-gen-ai-project/test.csv")

# -------------------------
# 2. Build Vocabulary
# -------------------------
def tokenize(text):
    return text.lower().split()

counter = Counter()
for text in train_df["text"]:
    counter.update(tokenize(text))

vocab = {word: idx + 2 for idx, (word, _) in enumerate(counter.items())}
vocab["<PAD>"] = 0
vocab["<UNK>"] = 1

def encode(text, vocab, max_len):
    tokens = tokenize(text)
    ids = [vocab.get(w, vocab["<UNK>"]) for w in tokens][:max_len]
    if len(ids) < max_len:
        ids += [vocab["<PAD>"]] * (max_len - len(ids))
    return ids

# -------------------------
# 3. Dataset & DataLoaders
# -------------------------
class EmotionDataset(Dataset):
    def __init__(self, df, vocab, mode="train", max_len=50):
        self.df = df
        self.vocab = vocab
        self.mode = mode
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text_ids = torch.tensor(
            encode(self.df.iloc[idx]["text"], self.vocab, self.max_len),
            dtype=torch.long
        )
        if self.mode == "train":
            labels = torch.tensor(
                self.df.iloc[idx][label_cols].values.astype("float32")
            )
            return text_ids, labels
        else:
            return text_ids

train_df_split, val_df_split = train_test_split(
    train_df,
    test_size=config.val_split,
    random_state=config.random_state
)

train_dataset = EmotionDataset(train_df_split, vocab, mode="train", max_len=config.max_len)
val_dataset   = EmotionDataset(val_df_split, vocab, mode="train", max_len=config.max_len)
test_dataset  = EmotionDataset(test_df, vocab, mode="test", max_len=config.max_len)

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=config.batch_size)
test_loader  = DataLoader(test_dataset, batch_size=config.batch_size)

# -------------------------
# 4. BiLSTM Model
# -------------------------
class BiLSTMEmotionNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )
        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x):
        x = self.embedding(x)
        _, (h_n, _) = self.lstm(x)
        h_forward = h_n[-2, :, :]
        h_backward = h_n[-1, :, :]
        h = torch.cat((h_forward, h_backward), dim=1)
        out = self.fc(h)
        return torch.sigmoid(out)

model = BiLSTMEmotionNN(
    vocab_size=len(vocab),
    embed_dim=config.embed_dim,
    hidden_dim=config.hidden_dim,
    output_dim=len(label_cols)
).to(device)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=config.lr)

# -------------------------
# 5. Early Stopping Helper
# -------------------------
class EarlyStopper:
    def __init__(self, patience=3, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False

    def __call__(self, score):
        if self.best_score is None or score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

early_stopper = EarlyStopper(
    patience=config.patience,
    min_delta=0.001
)

# -------------------------
# 6. Training Loop with Validation
# -------------------------
n_epochs = config.n_epochs
best_model_state = None

for epoch in range(n_epochs):
    model.train()
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_train_loss = running_loss / len(train_loader)

    # Validation
    model.eval()
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            preds = (outputs.cpu().numpy() > config.threshold).astype(int)
            all_preds.append(preds)
            all_labels.append(labels.numpy())

    y_true = np.vstack(all_labels)
    y_pred = np.vstack(all_preds)

    val_f1_macro = f1_score(y_true, y_pred, average="macro")
    val_accuracy = accuracy_score(y_true, y_pred)

    print(
        f"Epoch {epoch+1}/{n_epochs} "
        f"- Train Loss: {avg_train_loss:.4f} "
        f"- Val Macro F1: {val_f1_macro:.4f} "
        f"- Val Accuracy: {val_accuracy:.4f}"
    )

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": avg_train_loss,
        "val_f1_macro": val_f1_macro,
        "val_accuracy": val_accuracy
    })

    # Early stopping
    early_stopper(val_f1_macro)
    if early_stopper.early_stop:
        print(f"Early stopping triggered at epoch {epoch+1}")
        break
    else:
        best_model_state = model.state_dict()

# Restore best model if early stopped
if best_model_state is not None:
    model.load_state_dict(best_model_state)

# -------------------------
# 7. Test Predictions & Submission
# -------------------------
model.eval()
test_preds = []

with torch.no_grad():
    for inputs in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        preds = (outputs.cpu().numpy() > config.threshold).astype(int)
        test_preds.append(preds)

test_preds = np.vstack(test_preds)

submission = pd.DataFrame(test_preds, columns=label_cols)
submission.insert(0, "id", test_df["id"])

submission_filename = "submission.csv"
submission.to_csv(submission_filename, index=False)
print(f"Saved {submission_filename}")

wandb.finish()


Using device: cuda
Epoch 1/20 - Train Loss: 0.5837 - Val Macro F1: 0.1460 - Val Accuracy: 0.1230
Epoch 2/20 - Train Loss: 0.5418 - Val Macro F1: 0.2480 - Val Accuracy: 0.1596
Epoch 3/20 - Train Loss: 0.5050 - Val Macro F1: 0.3257 - Val Accuracy: 0.2020
Epoch 4/20 - Train Loss: 0.4491 - Val Macro F1: 0.4004 - Val Accuracy: 0.2504
Epoch 5/20 - Train Loss: 0.3821 - Val Macro F1: 0.4895 - Val Accuracy: 0.3221
Epoch 6/20 - Train Loss: 0.3151 - Val Macro F1: 0.5491 - Val Accuracy: 0.3675
Epoch 7/20 - Train Loss: 0.2564 - Val Macro F1: 0.5792 - Val Accuracy: 0.3997
Epoch 8/20 - Train Loss: 0.2106 - Val Macro F1: 0.6413 - Val Accuracy: 0.4363
Epoch 9/20 - Train Loss: 0.1725 - Val Macro F1: 0.6512 - Val Accuracy: 0.4817
Epoch 10/20 - Train Loss: 0.1395 - Val Macro F1: 0.6737 - Val Accuracy: 0.5139
Epoch 11/20 - Train Loss: 0.1126 - Val Macro F1: 0.6845 - Val Accuracy: 0.5373
Epoch 12/20 - Train Loss: 0.0931 - Val Macro F1: 0.7032 - Val Accuracy: 0.5608
Epoch 13/20 - Train Loss: 0.0760 - Val Mac

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_loss,█▇▇▆▅▅▄▃▃▂▂▂▂▁▁▁▁▁▁▁
val_accuracy,▁▂▂▃▄▅▅▆▆▇▇▇▇███████
val_f1_macro,▁▂▃▄▅▆▆▇▇▇██████████
epoch,20
train_loss,0.03188
val_accuracy,0.58419
val_f1_macro,0.71751


<hr/>

<h3 style="color:#D35400;">
  Results & Runtime Summary
</h3>

<p style="font-size:16px;">
  <strong>Model:</strong> PyTorch BiLSTM (from scratch) for multi-label emotion classification
</p>

<p style="font-size:16px;">
  <strong>Final Training Epoch:</strong> 20<br/>
  Train Loss: <strong>0.0319</strong>
</p>

<p style="font-size:16px;">
  <strong>Validation Metrics (Internal):</strong><br/>
  Macro F1 Score: <strong>0.7175</strong><br/>
  Accuracy: <strong>0.5842</strong>
</p>

<p style="font-size:16px;">
  <strong>Kaggle Public Leaderboard Score:</strong><br/>
  Macro F1 Score: <strong>0.714</strong>
</p>

<p style="font-size:16px;">
  <strong>Runtime Environment:</strong>
</p>

<ul style="font-size:16px; line-height:1.6;">
  <li><strong>Platform:</strong> Kaggle Notebook</li>
  <li><strong>Hardware:</strong> GPU (NVIDIA T4 &times; 2)</li>
  <li><strong>Estimated Runtime:</strong> ~10–20 minutes (including training and validation)</li>
  <li><strong>Frameworks:</strong> PyTorch, scikit-learn, pandas, NumPy</li>
</ul>


<p style="font-size:16px;">
  <strong>Observations:</strong>
</p>

<ul style="font-size:16px; line-height:1.6;">
  <li>
    The BiLSTM model achieves performance comparable to the MLP model,
    while operating directly on learned sequence representations instead
    of TF-IDF features.
  </li>
  <li>
    The small gap between validation Macro F1 and Kaggle score indicates
    that the model generalizes reasonably well to unseen test data.
  </li>
  <li>
    This experiment demonstrates the use of a fully custom PyTorch
    training loop with early stopping, laying the foundation for more
    advanced deep learning and transformer-based models.
  </li>
</ul>